# ViT Image Embedding — Amazon All Beauty Reviews
### Pipeline
1. Load pre-cleaned interactions from `cleaned_interactions.jsonl` (output of `preprocess.py`)
2. Apply image-specific validation (URL resolution, size checks, retry logic)
3. Embed each `(user, item)` review image with `openai/clip-vit-base-patch32`
4. Save one record per interaction to `interactions_image_embeddings.jsonl`

### Why CLIP instead of raw ViT?
Raw `google/vit-base-patch16-224` was trained on ImageNet for object classification.
Its embedding space has no relationship to language. CLIP was trained on 400M
image-text pairs with a contrastive objective that aligns visual and textual
representations into a shared semantic space — a prerequisite for the cross-modal
sentiment prototype stage where `vi` and `tui` must be projectable onto the same
prototype vectors `{s1, ..., sK}`.


In [1]:
!pip install transformers pillow tqdm requests

In [2]:
import json
import torch
import requests
import random
import numpy as np

from PIL import Image
from io import BytesIO
from tqdm import tqdm
from collections import defaultdict
from transformers import CLIPProcessor, CLIPModel

random.seed(42)
print("Imports OK")

Imports OK


In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
CLEAN_PATH    = "cleaned_interactions.jsonl"   # output of preprocess.py
OUTPUT_PATH   = "interactions_image_embeddings.jsonl"
CLIP_MODEL    = "openai/clip-vit-base-patch32"
MAX_USERS     = 500
MIN_REVIEWS   = 3    # must match preprocess.py
FETCH_TIMEOUT = 8    # seconds per image request
MAX_RETRIES   = 2    # retry failed image fetches once

random.seed(42)
print(f"Config loaded. Reading from : {CLEAN_PATH}")
print(f"CLIP model                  : {CLIP_MODEL}")

Config loaded. Reading from : cleaned_interactions.jsonl
CLIP model                  : openai/clip-vit-base-patch32


In [4]:
# ── Image-specific validation helpers ─────────────────────────────────────────
# Note: core field validation, deduplication, and user-level filtering are
# already handled by preprocess.py. These functions handle image-only concerns.

def get_best_image_url(images) -> str | None:
    """
    Extract the highest-quality non-null HTTP image URL from a review's
    images list.  Tries keys in descending quality order.
    Handles both list-of-dicts (review images) and plain string URLs.
    """
    if not images:
        return None

    priority = ["hi_res", "large_image_url", "large", "medium_image_url",
                "small_image_url", "thumb"]

    for img in images:
        if isinstance(img, dict):
            for key in priority:
                url = img.get(key)
                if url and isinstance(url, str) and url.startswith("http"):
                    return url
        elif isinstance(img, str) and img.startswith("http"):
            return img
    return None


def fetch_image(url: str, timeout: int, retries: int) -> Image.Image | None:
    """
    Fetch an image from a URL with retry logic.
    Returns a PIL Image in RGB mode, or None on failure.
    Rejects images smaller than 32x32 (likely placeholders or corrupt).
    """
    for attempt in range(retries):
        try:
            resp = requests.get(url, timeout=timeout)
            resp.raise_for_status()
            img = Image.open(BytesIO(resp.content)).convert("RGB")
            if img.size[0] < 32 or img.size[1] < 32:
                print(f"  [SKIP] Image too small {img.size}: {url}")
                return None
            return img
        except Exception as e:
            if attempt == retries - 1:
                print(f"  [WARN] Failed after {retries} attempts — {e}")
    return None


print("Image helpers defined.")
print("  get_best_image_url(images) → str | None")
print("  fetch_image(url, timeout, retries) → PIL.Image | None")

Image helpers defined.
  get_best_image_url(images) → str | None
  fetch_image(url, timeout, retries) → PIL.Image | None


In [5]:
# ── Load CLIP model ────────────────────────────────────────────────────────────
device    = "cuda" if torch.cuda.is_available() else "cpu"
processor = CLIPProcessor.from_pretrained(CLIP_MODEL)
model     = CLIPModel.from_pretrained(CLIP_MODEL).to(device)
model.eval()

# CLIP vision encoder CLS token dimension
EMB_DIM = model.config.vision_config.hidden_size
print(f"Device     : {device}")
print(f"Model      : {CLIP_MODEL}")
print(f"Embedding dim (vision CLS): {EMB_DIM}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Device     : cuda
Model      : openai/clip-vit-base-patch32
Embedding dim (vision CLS): 768


In [6]:
# ── Load pre-cleaned interactions and filter to users with image reviews ───────
raw_records = []
with open(CLEAN_PATH) as f:
    for line in f:
        line = line.strip()
        if line:
            raw_records.append(json.loads(line))

print(f"Clean interactions loaded : {len(raw_records)}")

# Keep only records that have at least one resolvable image URL
# (text-only records are skipped — the BERT notebook handles those)
image_records = [
    r for r in raw_records
    if get_best_image_url(r.get("images", [])) is not None
]
print(f"Records with usable image : {len(image_records)}")

# Group by user, enforce MIN_REVIEWS on image-bearing records, sample MAX_USERS
user_map = defaultdict(list)
for r in image_records:
    user_map[r["user_id"]].append(r)

qualified_uids = sorted(
    uid for uid, recs in user_map.items() if len(recs) >= MIN_REVIEWS
)
print(f"Qualified users (≥{MIN_REVIEWS} image reviews): {len(qualified_uids)}")

sampled_uids = random.sample(qualified_uids, min(MAX_USERS, len(qualified_uids)))
print(f"Users sampled for embedding : {len(sampled_uids)}")

total_candidates = sum(len(user_map[uid]) for uid in sampled_uids)
print(f"Total image interactions to attempt : {total_candidates}")

FileNotFoundError: [Errno 2] No such file or directory: 'cleaned_interactions.jsonl'

In [ ]:
# ── CLIP image embedding function ─────────────────────────────────────────────

def get_image_embedding(img: Image.Image) -> np.ndarray | None:
    """
    Embed a PIL image using the CLIP vision encoder CLS token.
    Returns L2-normalised np.ndarray of shape (EMB_DIM,), or None on error.

    We L2-normalise to match the prototype attention formula  vi^T sk,
    consistent with the text embeddings from the BERT notebook.
    """
    try:
        inputs = processor(images=img, return_tensors="pt").to(device)
        with torch.no_grad():
            vision_out = model.vision_model(**inputs)
        # CLS token — shape (1, EMB_DIM)
        emb = vision_out.last_hidden_state[:, 0, :].squeeze().cpu().numpy()
        # L2-normalise
        norm = np.linalg.norm(emb)
        if norm > 0:
            emb = emb / norm
        return emb.astype(np.float32)
    except Exception as e:
        print(f"  [ERROR] Embedding failed: {e}")
        return None


print("Embedding function defined.")
print(f"  get_image_embedding(PIL.Image) → np.ndarray shape ({EMB_DIM},)")

In [ ]:
# ── Generate per-interaction image embeddings ──────────────────────────────────
# We embed at the (user, item) interaction level — one embedding per review
# that has an image. The prototype module needs individual vi vectors, not a
# mean-pooled user vector, so that rating supervision can align each image's
# embedding to the correct sentiment prototype independently.

print("Fetching images and embedding…\n")

interaction_embs = []    # list of dicts — one per successfully embedded (user, asin)
fetch_ok  = 0
fetch_err = 0

for uid in tqdm(sampled_uids, desc="Users"):
    for r in user_map[uid]:
        url = get_best_image_url(r.get("images", []))
        if not url:
            continue

        img = fetch_image(url, FETCH_TIMEOUT, MAX_RETRIES)
        if img is None:
            fetch_err += 1
            continue

        emb = get_image_embedding(img)
        if emb is None:
            fetch_err += 1
            continue

        fetch_ok += 1
        interaction_embs.append({
            "user_id"        : uid,
            "asin"           : r.get("asin"),
            "parent_asin"    : r.get("parent_asin"),
            "rating"         : r.get("rating"),
            "timestamp"      : r.get("timestamp"),
            "image_url"      : url,
            "image_embedding": emb,    # np.ndarray (EMB_DIM,)
        })

print(f"\nImages fetched successfully : {fetch_ok}")
print(f"Images failed / skipped     : {fetch_err}")
print(f"Total interaction embeddings: {len(interaction_embs)}")

# Spot-check
if interaction_embs:
    ex = interaction_embs[0]
    print(f"\nSpot-check — first record:")
    print(f"  user_id  : {ex['user_id']}")
    print(f"  asin     : {ex['asin']}")
    print(f"  rating   : {ex['rating']}")
    print(f"  emb shape: {ex['image_embedding'].shape}")
    print(f"  emb norm : {np.linalg.norm(ex['image_embedding']):.4f}  (should be ~1.0)")

In [ ]:
# ── Save per-interaction image embeddings ─────────────────────────────────────
# Output schema (one JSON line per interaction):
# {
#   "user_id"        : str,
#   "asin"           : str,          ← JOIN KEY with text embeddings file
#   "parent_asin"    : str,
#   "rating"         : float,        ← supervision signal for prototype learning
#   "timestamp"      : int,
#   "image_url"      : str,          ← kept for debugging / inspection
#   "image_embedding": list[float],  ← len = EMB_DIM (768), L2-normalised
# }

with open(OUTPUT_PATH, "w") as f:
    for record in interaction_embs:
        out = {
            "user_id"        : record["user_id"],
            "asin"           : record["asin"],
            "parent_asin"    : record["parent_asin"],
            "rating"         : record["rating"],
            "timestamp"      : record["timestamp"],
            "image_url"      : record["image_url"],
            "image_embedding": record["image_embedding"].tolist(),
        }
        f.write(json.dumps(out) + "\n")

print(f"Saved {len(interaction_embs)} interaction records → {OUTPUT_PATH}")
print(f"Each record contains an image_embedding of dim {EMB_DIM}.")
print("Next step: run merge.py to join with interactions_text_embeddings.jsonl")